# 04 - Tabelas Gerenciadas vs Não Gerenciadas no Delta Lake

Este notebook analisa as diferenças entre **tabelas gerenciadas (managed)** e **tabelas não gerenciadas (unmanaged/external)** no contexto do Delta Lake com Apache Spark.

## Contexto: Os dois laboratórios

Nos notebooks anteriores trabalhamos com **dois padrões distintos de registro de tabelas**:

| Laboratório | Padrão | Descrição |
|-------------|--------|-----------|
| Notebook 02 | `df.write.format('delta').save(path)` | Grava dados no MinIO sem registrar no Catalog |
| Notebook 03 | `CREATE TABLE ... USING delta LOCATION '...'` | Registra tabela externa no Spark Catalog |

## Tipos de Tabelas

### Tabela Não Gerenciada (External/Unmanaged)
```sql
CREATE TABLE minha_tabela
USING delta
LOCATION 's3a://bronze/minha_tabela'  -- caminho explícito
```
- **Metadados:** gerenciados pelo Spark Catalog (Hive Metastore)
- **Dados:** armazenados no caminho especificado (MinIO, HDFS, etc.)
- **DROP TABLE:** remove apenas os metadados do Catalog; **os dados permanecem**

### Tabela Gerenciada (Managed)
```sql
CREATE TABLE minha_tabela
USING delta
-- sem LOCATION
```
- **Metadados + Dados:** ambos gerenciados pelo Spark (warehouse local)
- **DROP TABLE:** remove metadados E dados do disco/storage
- **Localização padrão:** `spark.sql.warehouse.dir` (geralmente `spark-warehouse/`)

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('Managed_vs_Unmanaged')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
print(f'Warehouse dir: {spark.conf.get("spark.sql.warehouse.dir")}')

---
## 2. Tabela NÃO GERENCIADA (External)

Criamos uma tabela externa apontando para dados já existentes no MinIO.
O Spark registra apenas os metadados no Catalog — os dados ficam no MinIO.

In [ ]:
# Limpar eventuais registros anteriores
spark.sql('DROP TABLE IF EXISTS ext_categoria')

# Criar tabela NÃO GERENCIADA (External): aponta para dados no MinIO
delta_path = f's3a://{BRONZE_BUCKET}/categoria'
spark.sql(f"""
    CREATE TABLE ext_categoria
    USING delta
    LOCATION '{delta_path}'
""")

print('Tabela ext_categoria (NÃO GERENCIADA) criada!')
print(f'LOCATION: {delta_path}\n')

spark.sql('SELECT * FROM ext_categoria ORDER BY cd_categoria').show()

In [ ]:
# Inspecionar metadados da tabela NÃO GERENCIADA
print('=== DESCRIBE EXTENDED da tabela NÃO GERENCIADA ===')
spark.sql('DESCRIBE EXTENDED ext_categoria').show(100, truncate=False)

**Observe nos metadados:**
- `Type: EXTERNAL` — confirma que é não gerenciada
- `Location: s3a://bronze/categoria` — dados ficam no MinIO
- `Provider: delta` — formato Delta Lake

---
## 3. Tabela GERENCIADA (Managed)

Criamos uma tabela gerenciada a partir dos mesmos dados.
O Spark armazena tanto metadados quanto dados no warehouse local (`spark-warehouse/`).

In [ ]:
# Limpar eventuais registros anteriores
spark.sql('DROP TABLE IF EXISTS mgd_categoria')

# Ler dados do MinIO
df_cat = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/categoria')

# Criar tabela GERENCIADA: sem LOCATION — dados vão para o warehouse local
df_cat.write.format('delta').mode('overwrite').saveAsTable('mgd_categoria')

print('Tabela mgd_categoria (GERENCIADA) criada no warehouse local!')
spark.sql('SELECT * FROM mgd_categoria ORDER BY cd_categoria').show()

In [ ]:
# Inspecionar metadados da tabela GERENCIADA
print('=== DESCRIBE EXTENDED da tabela GERENCIADA ===')
spark.sql('DESCRIBE EXTENDED mgd_categoria').show(100, truncate=False)

**Observe nos metadados:**
- `Type: MANAGED` — confirma que é gerenciada
- `Location: file://.../spark-warehouse/mgd_categoria` — dados no filesystem local
- `Provider: delta` — mesmo formato Delta Lake

---
## 4. Comparação lado a lado

In [ ]:
print('=== SHOW TABLES — Tabelas registradas no Catalog ===')
spark.sql('SHOW TABLES').show(truncate=False)

In [ ]:
# Extrair a localização de cada tabela
def get_table_location(table_name):
    rows = spark.sql(f'DESCRIBE EXTENDED {table_name}').collect()
    for row in rows:
        if row['col_name'] == 'Location':
            return row['data_type']
    return 'N/A'

def get_table_type(table_name):
    rows = spark.sql(f'DESCRIBE EXTENDED {table_name}').collect()
    for row in rows:
        if row['col_name'] == 'Type':
            return row['data_type']
    return 'N/A'

print(f'{"Tabela":<20} {"Tipo":<12} {"Localização dos Dados"}')
print('-' * 100)
for t in ['ext_categoria', 'mgd_categoria']:
    tp = get_table_type(t)
    loc = get_table_location(t)
    print(f'{t:<20} {tp:<12} {loc}')

---
## 5. Comportamento ao executar DROP TABLE

Esta é a **principal diferença operacional** entre os dois tipos.

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client('s3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1')

def contar_objetos_minio(bucket, prefix):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    return resp.get('KeyCount', 0)

print('Antes do DROP TABLE:')
n = contar_objetos_minio(BRONZE_BUCKET, 'categoria/')
print(f'  Objetos em s3://{BRONZE_BUCKET}/categoria/: {n}')

In [ ]:
# DROP na tabela NÃO GERENCIADA
print('--- DROP TABLE ext_categoria (NÃO GERENCIADA) ---')
spark.sql('DROP TABLE IF EXISTS ext_categoria')
print('DROP executado!')

# Verificar se os dados ainda existem no MinIO
n_depois = contar_objetos_minio(BRONZE_BUCKET, 'categoria/')
print(f'Objetos em s3://{BRONZE_BUCKET}/categoria/ após DROP: {n_depois}')

if n_depois > 0:
    print('✅ DADOS PRESERVADOS no MinIO — apenas metadados foram removidos do Catalog')
else:
    print('❌ Dados removidos')

In [ ]:
# DROP na tabela GERENCIADA
print('--- DROP TABLE mgd_categoria (GERENCIADA) ---')

# Verificar localização antes
warehouse_dir = spark.conf.get('spark.sql.warehouse.dir').replace('file:', '')
managed_path = os.path.join(warehouse_dir, 'mgd_categoria')
existe_antes = os.path.exists(managed_path)
print(f'Warehouse path: {managed_path}')
print(f'Diretório existe antes do DROP: {existe_antes}')

spark.sql('DROP TABLE IF EXISTS mgd_categoria')
print('DROP executado!')

existe_depois = os.path.exists(managed_path)
print(f'Diretório existe após DROP: {existe_depois}')

if not existe_depois:
    print('✅ DADOS REMOVIDOS — tabela gerenciada apaga dados e metadados')
else:
    print('Dados ainda presentes (possível com algumas versões de Spark)')

---
## 6. Recriando após DROP

Demonstra outra diferença: a tabela não gerenciada pode ser recriada apontando para os dados preservados.

In [ ]:
# Recriar a tabela externa apontando para os dados preservados no MinIO
print('Recriando ext_categoria após o DROP (dados ainda estão no MinIO):')
spark.sql(f"""
    CREATE TABLE ext_categoria
    USING delta
    LOCATION 's3a://{BRONZE_BUCKET}/categoria'
""")

spark.sql('SELECT * FROM ext_categoria ORDER BY cd_categoria').show()
print('✅ Tabela recriada com sucesso — dados históricos preservados!')

In [ ]:
# O histórico Delta também é preservado após DROP/recriar de tabela externa!
print('=== Histórico preservado mesmo após DROP/recriar ===')
spark.sql('DESCRIBE HISTORY ext_categoria').select('version','timestamp','operation').show()

---
## 7. Tabela Gerenciada vs Não Gerenciada — Análise Comparativa

| Característica | Tabela Gerenciada | Tabela Não Gerenciada |
|----------------|------------------|-----------------------|
| **Localização dos dados** | `spark-warehouse/` (local) | Caminho especificado (MinIO, HDFS, S3...) |
| **DROP TABLE** | Remove **dados + metadados** | Remove apenas **metadados** |
| **Governança de dados** | Spark controla tudo | Dados independentes do Catalog |
| **Portabilidade** | Baixa (dependente do warehouse) | Alta (dados em storage externo) |
| **Uso em produção** | Menos comum em lakehouse | **Padrão recomendado** em ambientes cloud |
| **Time Travel** | ✅ Suportado | ✅ Suportado |
| **ACID** | ✅ Suportado | ✅ Suportado |
| **Recriação sem perda de dados** | ❌ Dados perdidos no DROP | ✅ Dados preservados no MinIO/S3 |
| **Multi-engine access** | Difícil (warehouse local) | ✅ Fácil (path compartilhado) |

### Quando usar cada tipo?

**Tabelas Gerenciadas:**
- Ambientes de desenvolvimento/sandbox onde os dados são efêmeros
- Quando Spark controla totalmente o ciclo de vida dos dados
- Tabelas temporárias de processamento

**Tabelas Não Gerenciadas (External):**
- **Arquitetura Medalhão** (Landing, Bronze, Silver, Gold) em cloud
- Dados compartilhados entre múltiplos engines (Spark, Trino, Athena)
- Ambientes onde os dados devem sobreviver à recriação do Catalog
- **Cenário deste projeto:** dados no MinIO, Spark apenas processa

---
## 8. Experimento Bônus: Criar Tabela Gerenciada Delta com SQL

Também é possível criar tabelas gerenciadas diretamente via SQL (sem escrever dados primeiro).

In [ ]:
# Criar tabela gerenciada via CREATE TABLE AS SELECT (CTAS)
spark.sql('DROP TABLE IF EXISTS mgd_resumo_pedidos')

# Registrar a tabela pedido (não gerenciada) para usar no CTAS
spark.sql('DROP TABLE IF EXISTS ext_pedido')
spark.sql(f"""
    CREATE TABLE ext_pedido
    USING delta
    LOCATION 's3a://{BRONZE_BUCKET}/pedido'
""")

# CTAS: cria tabela gerenciada com agregação
spark.sql("""
    CREATE TABLE mgd_resumo_pedidos
    USING delta
    AS
    SELECT
        ds_status,
        COUNT(*) AS qt_pedidos,
        ROUND(SUM(vl_total), 2) AS vl_total,
        ROUND(AVG(vl_total), 2) AS vl_medio
    FROM ext_pedido
    GROUP BY ds_status
    ORDER BY qt_pedidos DESC
""")

print('=== Tabela gerenciada mgd_resumo_pedidos (CTAS) ===')
spark.sql('SELECT * FROM mgd_resumo_pedidos ORDER BY qt_pedidos DESC').show()

print('Tipo da tabela:')
spark.sql('DESCRIBE EXTENDED mgd_resumo_pedidos').filter("col_name IN ('Type', 'Location')").show(truncate=False)

In [ ]:
# Listagem final de todas as tabelas registradas
print('=== Tabelas no Spark Catalog ao final do notebook ===')
spark.sql('SHOW TABLES').show(truncate=False)

In [ ]:
# Limpeza
for t in ['ext_categoria', 'ext_pedido', 'mgd_resumo_pedidos']:
    spark.sql(f'DROP TABLE IF EXISTS {t}')
    print(f'  {t} removida')

spark.stop()
print('\nSparkSession finalizada.')